# TRL Toy PPO Example for RLHF Metrics

This notebook uses a **tiny language model** and a **toy reward function** so you can satisfy the assignment without starting from a large RLHF setup.

The goal is to log and interpret these curves:

- `reward`
- `KL`
- `policy loss`
- `entropy`
- `response length`

We use:

- `sshleifer/tiny-gpt2` as the policy model
- TRL's classic `PPOTrainer`
- a handcrafted reward that prefers upbeat words like `good`, `great`, `helpful`, and `love`

This is intentionally small and educational, not a production RLHF pipeline.

## 1. Install dependencies

Run this once. If the kernel asks for a restart after installation, restart it and continue from the next cell.

Important compatibility note:

- This notebook uses the classic notebook-friendly `PPOTrainer` flow from `trl==0.12.2`.
- On **Python 3.13**, that old stack can fail while building `tokenizers` from source.
- The most reliable path for this exact notebook is to run it in a **Python 3.12** environment.

If you stay on Python 3.13, you would usually want a newer TRL/Transformers stack, but then the PPO code in this notebook would likely need API changes too.

In [ ]:
# Recommended only inside a Python 3.12 environment.
# Uncomment the next line the first time you run this notebook.
# %pip install -q "trl==0.12.2" "transformers==4.41.2" "accelerate>=0.30.0" "datasets>=2.19.0" "pandas" "matplotlib" "sentencepiece"

## 2. Imports and setup

In [ ]:
import random
from statistics import mean

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import trange
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, create_reference_model

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 3. Toy prompt set

A small prompt pool is enough for this milestone. PPO will sample responses for these prompts and optimize them against our toy reward.

In [ ]:
prompts = [
    "Finish this upbeat sentence: Today I feel",
    "Write one cheerful line about learning: Learning RLHF is",
    "Complete this sentence with a positive tone: My project looks",
    "Write a short encouraging sentence: You can",
    "Continue this happy phrase: The weekend was",
    "Write a friendly response: Thanks for your help, it was",
    "Finish the sentence positively: This notebook is",
    "Complete this thought: Working with tiny models feels",
]

len(prompts)

## 4. Load a tiny policy model and reference model

In [ ]:
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name)
ref_model = create_reference_model(model)

ppo_config = PPOConfig(
    model_name=model_name,
    learning_rate=1e-5,
    batch_size=4,
    mini_batch_size=2,
    gradient_accumulation_steps=1,
    ppo_epochs=4,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

generation_kwargs = {
    "do_sample": True,
    "top_k": 0,
    "top_p": 1.0,
    "max_new_tokens": 24,
    "min_new_tokens": 4,
    "pad_token_id": tokenizer.eos_token_id,
}

ppo_trainer.accelerator.device

## 5. Define a toy reward function

This is our stand-in for a reward model. It gives higher reward to responses containing upbeat words and a small bonus for non-trivial length.

In [ ]:
positive_words = {
    "good", "great", "helpful", "love", "happy", "excellent",
    "amazing", "kind", "fun", "clear", "bright", "encouraging"
}

negative_words = {"bad", "awful", "hate", "worse", "boring", "sad"}

def normalize_word(token: str) -> str:
    return token.strip(".,!?;:'\"()[]{}").lower()

def toy_reward(text: str) -> float:
    words = [normalize_word(word) for word in text.split()]
    pos_hits = sum(word in positive_words for word in words)
    neg_hits = sum(word in negative_words for word in words)
    length_bonus = 0.25 if len(words) >= 6 else -0.25
    return float(pos_hits - neg_hits + length_bonus)

toy_reward("this is a great and helpful notebook")

## 6. Run PPO and log the required metrics

In [ ]:
def to_scalar(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu().item())
    if hasattr(value, "item"):
        return float(value.item())
    return float(value)

def pick_stat(stats, *keys, default=float("nan")):
    for key in keys:
        if key in stats:
            return to_scalar(stats[key])
    return default

num_updates = 30
history = []

for update in trange(num_updates, desc="PPO updates"):
    batch_prompts = random.sample(prompts, k=ppo_config.batch_size)
    query_tensors = []
    response_tensors = []
    rewards = []
    decoded_responses = []
    response_lengths = []

    for prompt in batch_prompts:
        query_tensor = tokenizer.encode(prompt, return_tensors="pt").squeeze(0)
        query_tensor = query_tensor.to(ppo_trainer.accelerator.device)
        query_tensors.append(query_tensor)

        response_tensor = ppo_trainer.generate(query_tensor, return_prompt=False, **generation_kwargs)
        response_tensor = response_tensor.squeeze(0)
        response_tensors.append(response_tensor)

        response_text = tokenizer.decode(response_tensor, skip_special_tokens=True)
        decoded_responses.append(response_text)
        response_lengths.append(int(response_tensor.shape[-1]))

        reward_value = toy_reward(response_text)
        rewards.append(torch.tensor(reward_value, dtype=torch.float32).to(ppo_trainer.accelerator.device))

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    history.append({
        "update": update + 1,
        "reward": mean([to_scalar(r) for r in rewards]),
        "kl": pick_stat(stats, "objective/kl", "ppo/policy/approxkl"),
        "policy_loss": pick_stat(stats, "ppo/loss/policy", "loss/policy_avg"),
        "entropy": pick_stat(stats, "ppo/policy/entropy", "objective/entropy", "policy/entropy_avg"),
        "response_length": mean(response_lengths),
        "sample_prompt": batch_prompts[0],
        "sample_response": decoded_responses[0],
    })

metrics_df = pd.DataFrame(history)
metrics_df.tail()

## 7. Plot the curves required by the assignment

In [ ]:
plot_columns = ["reward", "kl", "policy_loss", "entropy", "response_length"]
titles = ["Reward", "KL", "Policy Loss", "Entropy", "Response Length"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, column, title in zip(axes, plot_columns, titles):
    ax.plot(metrics_df["update"], metrics_df[column], marker="o", linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel("PPO update")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("value")
plt.tight_layout()
plt.show()

## 8. Inspect a few sampled responses

This is useful for connecting the metric curves to actual behavior.

In [ ]:
metrics_df[["update", "reward", "kl", "entropy", "response_length", "sample_prompt", "sample_response"]].tail(10)

## 9. What each curve means

Use these points in your assignment write-up:

- **Reward**: This is the average score from the toy reward function. If it trends upward, the policy is learning to generate responses that contain more reward-favored patterns.
- **KL**: KL measures how far the updated policy has moved away from the frozen reference model. A small or moderate KL is healthy. A rapidly exploding KL usually means the policy is drifting too aggressively.
- **Policy loss**: This is the PPO optimization objective for the policy network. It is often noisy and does not have to decrease smoothly every step. What matters is that it stays finite and training remains stable.
- **Entropy**: Entropy measures how random or diverse the token distribution is. Higher entropy means the model is exploring more. If entropy collapses too fast, the model may become overly deterministic too early.
- **Response length**: This tracks how long the generated completions are. It helps catch degenerate behavior like always producing very short outputs or growing to the max token limit just to chase reward.

A reasonable toy PPO run often looks like this:

- reward gradually increases
- KL stays controlled rather than exploding
- policy loss is noisy but finite
- entropy slowly decreases as the policy becomes more confident
- response length stays in a sensible range instead of collapsing

## 10. Short conclusion you can reuse

This notebook demonstrates a minimal RLHF-style PPO loop with TRL using a tiny language model and a handcrafted reward. Even though the reward is synthetic, the logged curves still illustrate the main RLHF training signals: the policy is pushed toward higher reward, constrained by KL to remain close to a reference model, while entropy and response length help diagnose exploration and collapse.

In [7]:
from trl import RewardTrainer, RewardConfig
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()

# Create the config with use_cpu=True
config = RewardConfig(
    output_dir="Qwen3-0.6B-Reward",
    use_cpu=True,
    logging_steps=10,           # Log every 10 steps
    logging_strategy="steps",   # Log based on steps rather than epochs
    log_level="info",           # Show info-level logs
    report_to=[],
    disable_tqdm=False,           # Ensure progress bar shows
    num_train_epochs=1,   
)

trainer = RewardTrainer(
    model="Qwen/Qwen3-0.6B",
    train_dataset=load_dataset("trl-lib/ultrafeedback_binarized", split="train"),
    args=config,  # Pass the config here
)

trainer.train()

KeyboardInterrupt: 

In [8]:
4%3

1